In [ ]:
import pandas as pd
import requests
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report

# =============================
# 1. LOAD DATASET
# =============================
df = pd.read_csv("irrigation_prediction[1].csv")

print("Dataset Loaded Successfully")
print(df.head())

# =============================
# 2. ENCODE CATEGORICAL DATA
# =============================
label_encoders = {}

for col in df.columns:
    if df[col].dtype == "object":
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col])
        label_encoders[col] = le

# =============================
# 3. SPLIT DATA
# =============================
X = df.drop("Irrigation_Need", axis=1)
y = df["Irrigation_Need"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# =============================
# 4. TRAIN MODEL
# =============================
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

# =============================
# 5. CHECK ACCURACY
# =============================
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print("\nModel Accuracy:", accuracy * 100)
print(classification_report(y_test, y_pred))

# =============================
# 6. WEATHER API FUNCTION
# =============================
def get_weather(city, api_key):
    url = f"https://api.openweathermap.org/data/2.5/weather?q={city}&appid={api_key}&units=metric"

    response = requests.get(url)
    data = response.json()

    if response.status_code == 200:
        temp = data["main"]["temp"]
        humidity = data["main"]["humidity"]
        rainfall = data.get("rain", {}).get("1h", 0)

        return temp, humidity, rainfall
    else:
        print("API Error:", data)
        return None

# =============================
# 7. LIVE PREDICTION FUNCTION
# =============================
def predict_irrigation_live(city, api_key):
    weather = get_weather(city, api_key)

    if weather is None:
        return

    temp, humidity, rainfall = weather

    print(f"\nLive Weather for {city}")
    print("Temperature:", temp)
    print("Humidity:", humidity)
    print("Rainfall:", rainfall)

    # sample input based on dataset structure
    sample = X.iloc[0].copy()

    # replace weather-related fields
    if "Temperature_C" in sample.index:
        sample["Temperature_C"] = temp

    if "Humidity" in sample.index:
        sample["Humidity"] = humidity

    if "Rainfall_mm" in sample.index:
        sample["Rainfall_mm"] = rainfall

    sample_df = pd.DataFrame([sample])

    prediction = model.predict(sample_df)

    result = label_encoders["Irrigation_Need"].inverse_transform(prediction)

    print("\nPredicted Irrigation Need:", result[0])

    if result[0] == "High":
        print("Recommendation: Immediate irrigation required")
    elif result[0] == "Medium":
        print("Recommendation: Moderate irrigation")
    else:
        print("Recommendation: No irrigation needed now")

# =============================
# 8. RUN
# =============================



Dataset Loaded Successfully
  Soil_Type  Soil_pH  Soil_Moisture  Organic_Carbon  Electrical_Conductivity  \
0      Clay     6.14          36.48            0.42                     2.17   
1      Silt     6.41          50.56            0.38                     0.23   
2     Sandy     7.71          40.07            1.09                     2.18   
3      Clay     5.96          12.75            1.56                     0.40   
4      Clay     7.76          18.58            0.95                     2.52   

   Temperature_C  Humidity  Rainfall_mm  Sunlight_Hours  Wind_Speed_kmh  \
0          21.90     31.19      1167.70            4.01            1.97   
1          36.50     26.01       831.28           10.72           16.82   
2          41.83     76.41      1844.45            7.75           19.03   
3          37.22     43.32       306.26            8.90           11.44   
4          22.38     86.44      1875.63           10.39           11.26   

  Crop_Type Crop_Growth_Stage  Season Ir

In [ ]:
import requests

API_KEY = "118626634a03534649aba304c8d5e54e"

url = f"https://api.openweathermap.org/data/2.5/weather?lat=44.34&lon=10.99&appid={API_KEY}&units=metric"

response = requests.get(url)

print("Status Code:", response.status_code)
print(response.json())

Status Code: 401
{'cod': 401, 'message': 'Invalid API key. Please see https://openweathermap.org/faq#error401 for more info.'}
